In [ ]:
# Les dépendances sont installées via requirements.txt
# Pour réinstaller : pip install -r ../requirements.txt


# 🧪 Étape 6 : Évaluation & Analyse des Anomalies de Succès

Cette étape évalue le **modèle unifié** de l'étape 5 selon deux axes :
1. **Métriques de performance** : R², MAE, RMSE sur le rang relatif par genre + validation croisée 5-fold
2. **Analyse des anomalies** : identifier les morceaux dont le succès (ou l'échec) ne peut pas être expliqué par le modèle — ce sont eux qui révèlent les vrais leviers non-audio (marketing, viralité, prestige artiste)

### 1. Préparation de l'environnement

In [2]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

# --- Même pipeline que 05_modelisation ---
AUDIO_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

df = pd.read_csv('../data/processed/cleaned_data_sample.csv')

df['artist_reputation'] = df.groupby('artists')['popularity'].transform('mean')
df['pop_rank_in_genre'] = df.groupby('track_genre')['popularity'].transform(
    lambda x: x.rank(pct=True)
)

scaler = StandardScaler()
audio_scaled = scaler.fit_transform(df[AUDIO_FEATURES])
audio_scaled_df = pd.DataFrame(audio_scaled, columns=AUDIO_FEATURES, index=df.index)
audio_scaled_df['track_genre'] = df['track_genre'].values
genre_centroids = audio_scaled_df.groupby('track_genre')[AUDIO_FEATURES].mean()
centroid_matrix = np.array([genre_centroids.loc[g].values for g in df['track_genre']])
df['genre_distance'] = np.linalg.norm(audio_scaled - centroid_matrix, axis=1)

df['duration_min'] = df['duration_ms'] / 60000
df['explicit_int']  = df['explicit'].astype(int)

MODEL_FEATURES = AUDIO_FEATURES + ['artist_reputation', 'genre_distance', 'duration_min', 'explicit_int']
TARGET = 'pop_rank_in_genre'

print(f"Pipeline prêt : {len(df):,} morceaux × {len(MODEL_FEATURES)} features")
print("Librairies prêtes pour l'évaluation du modèle unifié !")

Pipeline prêt : 97,270 morceaux × 13 features
Librairies prêtes pour l'évaluation du modèle unifié !


### 2. Évaluation sur Split Train/Test

On ré-entraîne le modèle sur 80% des données et on l'évalue sur les 20% restants. La cible est le **rang relatif dans le genre** (0-1), pas la popularité brute — ce qui donne des métriques bien plus interprétables.

In [3]:
X = df[MODEL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

resultats = pd.DataFrame({
    'Métrique': ['MAE', 'RMSE', 'R²'],
    'Valeur': [round(mae, 4), round(rmse, 4), round(r2, 3)],
    'Interprétation': [
        f'Erreur moyenne de ±{mae:.3f} sur le rang 0-1 (soit ±{mae*100:.1f} positions centiles)',
        'Erreur quadratique — pénalise les grosses erreurs de rang',
        f'{r2*100:.1f}% de la variance du rang relatif expliquée par le modèle'
    ]
})
display(resultats)

# Comparaison sur split aléatoire identique (random_state=42)
# Ancien modèle (popularité brute, split séquentiel non aléatoire) : R²=0.230
# Note : les cibles sont différentes (rang 0-1 vs popularité 0-100), les R² ne sont pas directement comparables
print(f"\nNouveau modèle — R² (split aléatoire 80/20) : {r2:.3f}")
print(f"Ancien modèle  — R² (split séquentiel 80/20) : 0.230")
print(f"\n→ Cibles différentes (rang relatif vs popularité brute) — voir validation croisée pour comparaison équitable.")

,Métrique,Valeur,Interprétation
0,MAE,0.1956,Erreur moyenne de ±0.196 sur le rang 0-1 (soit...
1,RMSE,0.2389,Erreur quadratique — pénalise les grosses erre...
2,R²,0.3100,31.0% de la variance du rang relatif expliquée...



Nouveau modèle — R² (split aléatoire 80/20) : 0.310
Ancien modèle  — R² (split séquentiel 80/20) : 0.230

→ Cibles différentes (rang relatif vs popularité brute) — voir validation croisée pour comparaison équitable.


### 3. Validation Croisée 5-Fold

La validation croisée garantit que les résultats ne dépendent pas du découpage aléatoire. Chaque fold entraîne un modèle différent sur 4/5 des données et le teste sur 1/5 restant.

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores, rmse_scores, r2_scores = [], [], []

print("=== Validation Croisée 5-Fold — Modèle Unifié ===\n")

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    m = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1)
    m.fit(X_tr, y_tr)
    p = m.predict(X_te)

    mae_scores.append(mean_absolute_error(y_te, p))
    rmse_scores.append(np.sqrt(mean_squared_error(y_te, p)))
    r2_scores.append(r2_score(y_te, p))
    print(f"Fold {fold+1} → MAE: {mae_scores[-1]:.4f} | RMSE: {rmse_scores[-1]:.4f} | R²: {r2_scores[-1]:.3f}")

print(f"\n=== Résultats Moyens sur 5 Folds ===")
cv_results = pd.DataFrame({
    'Métrique': ['MAE', 'RMSE', 'R²'],
    'Moyenne':  [round(np.mean(mae_scores), 4), round(np.mean(rmse_scores), 4), round(np.mean(r2_scores), 3)],
    'Écart-type': [round(np.std(mae_scores), 4), round(np.std(rmse_scores), 4), round(np.std(r2_scores), 3)]
})
display(cv_results)
print("\nValidation croisée documentée avec succès !")

=== Validation Croisée 5-Fold — Modèle Unifié ===



Fold 1 → MAE: 0.1955 | RMSE: 0.2388 | R²: 0.311


Fold 2 → MAE: 0.1957 | RMSE: 0.2383 | R²: 0.314


Fold 3 → MAE: 0.1946 | RMSE: 0.2373 | R²: 0.318


Fold 4 → MAE: 0.1952 | RMSE: 0.2378 | R²: 0.318


Fold 5 → MAE: 0.1948 | RMSE: 0.2372 | R²: 0.326

=== Résultats Moyens sur 5 Folds ===


,Métrique,Moyenne,Écart-type
0,MAE,0.1951,0.0004
1,RMSE,0.2379,0.0006
2,R²,0.3170,0.0050



Validation croisée documentée avec succès !


### 4. Analyse des Anomalies de Succès

Le modèle prédit bien les morceaux "normaux", mais certains **échouent complètement à la prédiction**. Ces résidus élevés sont les cas les plus riches en insights :

- **Résidu positif** (rang réel >> rang prédit) → **Surprise hit** : le morceau a explosé dans son genre malgré des caractéristiques audio ou un artiste sans historique fort. Signal viral, marketing, placement algorithmique.
- **Résidu négatif** (rang réel << rang prédit) → **Déception** : bonnes caractéristiques audio + artiste réputé, mais le morceau n'a pas percé. Mauvais timing, niche non exposée, concurrence forte.

Ces anomalies constituent la preuve empirique que **55% du succès musical reste inexplicable par les données audio seules**.

In [ ]:
# ── Construction du dataframe de test + résidus ─────────────────────────────
test_idx_arr     = y_test.index
df_test          = df.loc[test_idx_arr].copy()
df_test['predicted_rank'] = y_pred
df_test['actual_rank']    = y_test.values
df_test['residual']       = df_test['actual_rank'] - df_test['predicted_rank']

seuil_surprise  = df_test['residual'].quantile(0.95)
seuil_deception = df_test['residual'].quantile(0.05)
top_surprises   = df_test[df_test['residual'] >= seuil_surprise]
top_deceptions  = df_test[df_test['residual'] <= seuil_deception]

# ── Graphiques ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_test['predicted_rank'], df_test['actual_rank'],
                alpha=0.12, color='#535353', s=4, label=f'Morceaux ({len(df_test):,})')
axes[0].plot([0, 1], [0, 1], 'r--', lw=1.5, label='Prédiction parfaite')
axes[0].scatter(top_surprises['predicted_rank'],  top_surprises['actual_rank'],
                color='#1DB954', s=18, zorder=5, label=f'Surprises (n={len(top_surprises)})')
axes[0].scatter(top_deceptions['predicted_rank'], top_deceptions['actual_rank'],
                color='#ff6b6b', s=18, zorder=5, label=f'Déceptions (n={len(top_deceptions)})')
axes[0].set_xlabel("Rang prédit")
axes[0].set_ylabel("Rang réel dans le genre")
axes[0].set_title("Rang prédit vs Rang réel")
axes[0].legend(fontsize=8)

axes[1].hist(df_test['residual'], bins=60, color='#535353', edgecolor='none', alpha=0.8)
axes[1].axvline(x=0,               color='red',     ls='--', lw=1.5, label='Erreur nulle')
axes[1].axvline(x=seuil_surprise,  color='#1DB954', ls='--', lw=1.5, label=f'Surprises > {seuil_surprise:.2f}')
axes[1].axvline(x=seuil_deception, color='#ff6b6b', ls='--', lw=1.5, label=f'Déceptions < {seuil_deception:.2f}')
axes[1].set_xlabel("Résidu (rang réel − rang prédit)")
axes[1].set_title("Distribution des résidus")
axes[1].legend(fontsize=8)

plt.suptitle("Analyse des Anomalies de Succès", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ── Tables surprises / déceptions ────────────────────────────────────────────
COLS = ['track_name', 'artists', 'track_genre', 'popularity',
        'artist_reputation', 'predicted_rank', 'actual_rank', 'residual']

print("=" * 70)
print("TOP 20 SURPRISES — Hits que le modèle n'a pas vu venir")
print("=" * 70)
display(df_test.nlargest(20, 'residual')[COLS].round(3).reset_index(drop=True))

print("\n" + "=" * 70)
print("TOP 20 DÉCEPTIONS — Morceaux qui auraient dû performer")
print("=" * 70)
display(df_test.nsmallest(20, 'residual')[COLS].round(3).reset_index(drop=True))

# ── Profil comparatif ────────────────────────────────────────────────────────
profil = pd.DataFrame({
    'Groupe': ['Surprises (top 5%)', 'Déceptions (bottom 5%)', 'Tous'],
    'artist_reputation (moy)': [top_surprises['artist_reputation'].mean(),
                                 top_deceptions['artist_reputation'].mean(),
                                 df_test['artist_reputation'].mean()],
    'popularité réelle (moy)': [top_surprises['popularity'].mean(),
                                 top_deceptions['popularity'].mean(),
                                 df_test['popularity'].mean()],
}).round(2)
display(profil)
print(f"\nZone surprises  : {len(top_surprises)} morceaux")
print(f"Zone déceptions : {len(top_deceptions)} morceaux")